### Average stats_train_inputs (Male & Female)

In [ ]:
path = '/home/nashah/projects/bodies-at-rest/stats_train_inputs_processed.xlsx'

import pandas as pd
def load_data(path, sheet_name):
	df = pd.read_excel(path, sheet_name=sheet_name)
	return df

m_straight_limbs = load_data(path, 'm_straight_limbs')
m_straight_limbs_min	= m_straight_limbs['min']
m_straight_limbs_max	= m_straight_limbs['max']
m_straight_limbs_mean	= m_straight_limbs['mean']
m_straight_limbs_std_dev= m_straight_limbs['std_dev']

print(f"m_straight_limbs:\n{m_straight_limbs.head()}\n")

f_straight_limbs = load_data(path, 'f_straight_limbs')
f_straight_limbs_min	= f_straight_limbs['min']
f_straight_limbs_max	= f_straight_limbs['max']
f_straight_limbs_mean	= f_straight_limbs['mean']
f_straight_limbs_std_dev= f_straight_limbs['std_dev']

print(f"f_straight_limbs:\n{f_straight_limbs.head()}\n")

avg_min		= (m_straight_limbs_min		+ f_straight_limbs_min)		/ 2
avg_max		= (m_straight_limbs_max		+ f_straight_limbs_max)		/ 2
avg_mean	= (m_straight_limbs_mean	+ f_straight_limbs_mean)	/ 2
avg_std_dev	= (m_straight_limbs_std_dev	+ f_straight_limbs_std_dev)	/ 2

print(f"avg_min:\n{avg_min}\n")
print(f"avg_max:\n{avg_max}\n")
print(f"avg_mean:\n{avg_mean}\n")
print(f"avg_std_dev:\n{avg_std_dev}\n")

### Average stats_train_labels (Male & Female) ->
### Save to a new sheet in the same file

In [ ]:
path = '/home/nashah/projects/bodies-at-rest/stats_train_labels_processed_straight_limbs.xlsx'

import pandas as pd
def load_data(path, sheet_name):
    return pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')

# Load male and female sheets
m_straight_limbs = load_data(path, 'm_straight_limbs')
f_straight_limbs = load_data(path, 'f_straight_limbs')

# Compute averages
avg_min      = (m_straight_limbs['min']      + f_straight_limbs['min'])      / 2
avg_max      = (m_straight_limbs['max']      + f_straight_limbs['max'])      / 2
avg_mean     = (m_straight_limbs['mean']     + f_straight_limbs['mean'])     / 2
avg_std_dev  = (m_straight_limbs['std_dev']  + f_straight_limbs['std_dev'])  / 2

# Create DataFrame
averages_df = pd.DataFrame({
    'min': avg_min,
    'max': avg_max,
    'mean': avg_mean,
    'std_dev': avg_std_dev
})

# Append new sheet using openpyxl engine
with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    averages_df.to_excel(writer, sheet_name='avg_straight_limbs', index=False)

### Extract Standard Deviations

In [ ]:
import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ——— Load per-dimension std-devs from your Excel stats sheet ———
stats_path = "/home/nashah/projects/bodies-at-rest/stats_train_labels_processed_straight_limbs.xlsx"
avg = pd.read_excel(stats_path, sheet_name="avg_straight_limbs", engine="openpyxl")
avg_std = torch.tensor(avg["std_dev"].values, dtype=torch.float32, device=device)

# slice & reshape to match pred-tensor shapes
joints_std        = avg_std[   0:   72].reshape(1, 24, 3)   # 24 joints × (x,y,z)
betas_std         = avg_std[  72:   82].reshape(1, 10)      # 10 shape coefs
global_orient_std = avg_std[  82:   85].reshape(1,  3)      # root axis-angle
body_pose_std     = avg_std[  85:  154].reshape(1, 23, 3)   # 23 joints × (x,y,z)
transl_std        = avg_std[ 154:  157].reshape(1,  3)      # root translation (x,y,z)

print(f"joints_std:\n{joints_std.shape}\n")
print(f"betas_std:\n{betas_std.shape}\n")
print(f"global_orient_std:\n{global_orient_std.shape}\n")
print(f"body_pose_std:\n{body_pose_std.shape}\n")
print(f"transl_std:\n{transl_std.shape}\n")